### Retina Blood Vessels Segmentation using U-Net

#### Import Libraries

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt

##### Check if GPU is available

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#### Dataset for Retina Blood Vessels Segmentation

In [3]:
# link to the Dataset
# https://www.kaggle.com/datasets/abdallahwagih/retina-blood-vessel

In [4]:
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None, target_transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform                
        self.target_transform = target_transform 
        self.image_files = os.listdir(image_dir)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            mask = self.target_transform(mask)

        return image, mask


#### Transform the Images and Masks to Tensors

In [5]:
image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

mask_transform = transforms.Compose([
    transforms.ToTensor()
])

#### Local paths for the Dataset

In [6]:
train_dataset = SegmentationDataset(
    image_dir = './data/train/augmented/image',
    mask_dir = './data/train/augmented/mask',
    transform=image_transform,
    target_transform=mask_transform
)

val_dataset = SegmentationDataset(
    image_dir = './data/val/image',
    mask_dir = './data/val/mask',
    transform=image_transform,
    target_transform=mask_transform
)

test_dataset = SegmentationDataset(
    image_dir = './data/test/image',
    mask_dir = './data/test/mask',
    transform=image_transform,
    target_transform=mask_transform
)

#### Load the Dataset

In [7]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=10, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False)

### U-Net Model

U-Net is a type of a neural network which is mainly used for image segmentation i.e dividing the objects in an image into different parts for example separating a tumor from healthy tissue in a medical scan. The name “U-Net” comes from the shape of its architecture because the whole architecture resembles the shape of letter "U". It is widely used in medical imaging because it performs well even with a small amount of labeled data.

In [11]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [12]:
# Input Image:        3 channels (RGB)

# Down Block 1 →      64 features
# Down Block 2 →      128 features
# Down Block 3 →      256 features
# Down Block 4 →      512 features

# Bottleneck   →      1024 features (512 * 2)

# Up Block 1   ←      512 features
# Up Block 2   ←      256 features
# Up Block 3   ←      128 features
# Up Block 4   ←      64 features

# Final Output →      1 channel (mask)
    
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super(UNet, self).__int__()
        self.down_path = nn.ModuleList()
        self.up_path = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # DownSample Part
        for feature in features:
            self.down_path.append(DoubleConv(in_channels, feature))
            in_channels = feature

        # UpSample Part
        for feature in reversed(features):
            self.up_path.append(
                nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2)
            )
            self.up_path.append(DoubleConv(feature*2, feature))

        # BottleNeck Part
        self.bottle_neck = DoubleConv(features[-1], features[-1]*2)

        # Final Conv Layer 
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        for down in self.down_path:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottle_neck(x)
        skip_connections = skip_connections[::-1] # inverse the skip connections for concat

        for idx in range(0, len(self.up_path), 2):

            # for the upsample part as up_path[0]..[2]..gives upsample part from corresponding [0//2]..[2//2] block
            x = self.up_path[idx](x)
            skip_connections = skip_connections[idx//2]

            if x.shape != skip_connections.shape:
                x = torch.nn.functional.interpolate(x, size=skip_connections[2:])

            # concat the x from upsample op with skip connection as such that C+C=2C such that in_channels = 2*feature
            x = torch.cat((x, skip_connections), dim=1)

            # does the doub_conv as the pair with [1]..[3] gives conv part in the up_path module for [1//2]..[3//2] block
            x = self.up_path[idx+1](x)

        return torch.sigmoid(self.final_conv(x))       

